# PPO en Frogger (ALE): experimento educativo

**Objetivo:** construir un pipeline completo de *reinforcement learning* sobre el entorno
`ALE/Frogger-v5` usando `stable-baselines3` (PPO + CNN) y responder tres preguntas con
datos, no con opiniones:

1. ¿Cuánto retorno bruto por episodio logra una **política aleatoria**?
2. ¿Cuánto logra un **PPO sin entrenar** (mismos pesos que al construirlo)?
3. ¿Cuánto logra **el mismo PPO después de entrenar** (según el presupuesto de pasos)?

El entorno, el preprocesamiento y la evaluación viven en el helper `frogger_rl.py` del
proyecto independiente `ppo-frogger` (con sus propias pruebas unitarias). Este notebook
**usa** ese helper; no reimplementa el algoritmo ni el framework.

> **Nota importante.** Este proyecto usa **ALE (Atari 2600)**. *Atari no es Sega*: si la
> consigna de la clase exige específicamente **Genesis** (*stable-retro*, como el proyecto
> Airstriker en `ppo-sega`), este notebook es un complemento educativo, **no** un
> reemplazo de esa consigna.

Referencias oficiales: [ALE Frogger](https://ale.farama.org/environments/frogger/),
[ALE getting started](https://ale.farama.org/),
[ALE README (repositorio)](https://github.com/Farama-Foundation/Arcade-Learning-Environment),
[Gymnasium `AtariPreprocessing`](https://gymnasium.farama.org/api/wrappers/misc_wrappers/#gymnasium.wrappers.AtariPreprocessing),
[SB3 PPO](https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html).

## El juego, la observación y la parcial observabilidad

**El juego:** una rana debe cruzar una carretera con vehículos y un río con troncos hasta
llegar a alguna casilla de la fila superior. El **estado subyacente** del emulador es la
posición de cada actor (rana, vehículos, troncos), vidas, contador, etc.

**La observación:** el emulador entrega fotogramas RGB de 210×160×3; el helper los
convierte a grises de 84×84 y apila las **cuatro observaciones procesadas más
recientes** (ver la próxima sección). Eso hace que el problema sea
**parcialmente observable (POMDP)**:

- No vemos directamente la posición ni la velocidad de vehículos y troncos.
- Estados distintos pueden producir la misma pantalla (vidas, contador, temporizador
  no aparecen como variables separadas).
- El apilado de observaciones reduce la ambigüedad **temporal** de un solo fotograma.
- Las acciones "pegajosas" (`repeat_action_probability`) **no** mitigan la parcial
  observabilidad: añaden estocasticidad y desalientan secuencias de acciones
  memorizadas, pero no revelan el estado interno del emulador.

Formalmente: el agente ve $o_t \in \mathbb{R}^{4\times 84\times 84}$, una función parcial
del estado $s_t$ del emulador, y aprende una política $\pi_\theta(a_t \mid o_t)$ junto
con una función de valor $V_\theta(o_t)$.

## Acciones y recompensa nativa

`frogger_rl.ACTION_NAMES` define las **cinco acciones** del *minimal action set* de ALE
para Frogger (índices 0..4):

| Índice | Nombre | Efecto |
|---|---|---|
| 0 | `NOOP` | No hacer nada |
| 1 | `UP` | Subir |
| 2 | `RIGHT` | Derecha |
| 3 | `LEFT` | Izquierda |
| 4 | `DOWN` | Bajar |

**Recompensa nativa.** Usamos la recompensa cruda que devuelve el emulador: **no** la
recortamos ni la re-modelamos (sin `RewardClipping`, sin `VecNormalize`, sin
bonificaciones propias). Consecuencias honestas de esa decisión:

- **No prometemos** una recompensa negativa explícita por morir: la señal es la que ALE
  produce realmente (principalmente puntos por avanzar/llegar).
- Los retornos por episodio se suman tal cual; las pruebas del helper verifican esa suma
  con valores fuera del rango típico `[-1, 1]`.
- No usamos *terminal on life loss*: un episodio termina cuando el **juego** termina
  (`terminated`) o cuando se agota el **límite de pasos** (`truncated`); nunca
  confundimos una truncación por límite con una victoria.

## Preprocesamiento y contexto temporal de cuatro fotogramas

El helper `frogger_rl.make_env` aplica la receta canónica de Gymnasium/ALE:

1. `ALE/Frogger-v5` con `frameskip=1` y `repeat_action_probability=0.25` (acciones
   "pegajosas" / *sticky actions*, configuradas una única vez en esa capa).
2. Un solo `AtariPreprocessing(frame_skip=4, screen_size=84, grayscale_obs=True,
   scale_obs=False, terminal_on_life_loss=False)`: cada **paso del agente** repite la
   misma acción durante 4 cuadros del emulador y entrega **una** observación procesada
   de 84×84 en grises `uint8` (muestreada al final de esa repetición).
3. `FrameStackObservation(stack_size=4)`: apila las **últimas cuatro observaciones
   procesadas** en `(4, 84, 84)` `uint8` (canal primero, lista para CNN). Al iniciar
   un episodio el historial se rellena con el fotograma inicial (puede repetir solo
   ese); durante el juego normal **no** son cuatro copias del mismo cuadro.

**Muestreo:** las observaciones apiladas se toman en los **límites de repetición de la
acción** (una por paso de agente), no son cuatro fotogramas crudos consecutivos; el
fotograma crudo que produce ALE es RGB 210×160×3.
4. `TimeLimit(max_episode_steps=...)` medido en **pasos de agente** (el wrapper más
   externo). El default del helper es 2700 pasos; **no** heredamos el tope 450 de
   otros experimentos.

**¿Por qué 4 fotogramas?** Con un único frame, acciones como *izquierda/derecha* son
ambiguas (¿se mueve la rana o espera?). Cuatro fotogramas consecutivos revelan
**movimiento y dirección** — información esencial para estimar velocidades relativas de
vehículos y troncos, y para que la CNN detecte patrones temporales.

## PPO y la red CNN en una página

**Actor-crítico.** PPO entrena dos cabezas sobre la misma representación:

- **Actor** $\pi_\theta(a \mid o)$: distribución sobre las 5 acciones.
- **Crítico** $V_\theta(o)$: valor esperado del retorno desde la observación $o$.

**Ventaja.** Para saber si una acción fue mejor o peor que lo típico, PPO estima
$A_t = Q(s_t, a_t) - V(s_t)$ con GAE (`gae_lambda=0.95`) y descuento
(`gamma=0.99`). Si $A_t > 0$, esa acción fue mejor que el promedio de la política.

**Recorte (clip).** En cada repaso interno (`n_epochs`) PPO actualiza $\theta$ con
mini-lotes (`batch_size`) y recorta la **contribución del objetivo surrogate** (la
razón de importancia $\pi_\theta/\pi_{\theta_{old}}$) dentro del radio
`clip_range`. Es un límite sobre la pieza optimizada, **no** una garantía de que el
cambio efectivo de la política sea pequeño: la política puede cambiar más de lo que
sugiere el clip.

**CNN.** La observación `(4, 84, 84)` entra a la `NatureCNN` de SB3: capas
convolucionales que detectan bordes, luego patrones locales (vehículos, troncos) y
finalmente abstracciones de escena; el vector aplanado alimenta a las cabezas de actor
y crítico. Con `ent_coef` pequeño se mantiene algo de exploración y con `vf_coef` se
pondera la pérdida del valor.

**Hiperparámetros (modo normal):** `n_steps=512`, `batch_size=64`, `n_epochs=4`,
`learning_rate=2.5e-4`, `gamma=0.99`, `gae_lambda=0.95`, `clip_range=0.2`,
`ent_coef=0.01`, `vf_coef=0.5`, `device="cpu"` (igual que el proyecto original).

In [ ]:
# --- Preparación: entorno, raíz del proyecto y rutas ---------------------------------
import os
import sys
from pathlib import Path

import numpy as np

# Variables de ejecución (sin rutas absolutas hardcodeadas del workspace).
SMOKE = os.environ.get("FROGGER_SMOKE", "").strip().lower() in {"1", "true", "yes"}
TRAIN_SEED = int(os.environ.get("TRAIN_SEED", "2026"))
OUTPUT_ROOT = Path(os.environ["FROGGER_OUTPUT_DIR"]).resolve() if os.environ.get("FROGGER_OUTPUT_DIR") else None

# Detección de la raíz del proyecto: exige el helper frogger_rl.py. No se conforma
# con un pyproject.toml arbitrario de la raíz del workspace.
def find_project_root(start=None):
    here = Path(start or Path.cwd()).resolve()
    for candidate in [here, here / "ppo-frogger", *here.parents]:
        candidate = candidate.resolve()
        if (candidate / "frogger_rl.py").is_file():
            return candidate
    raise RuntimeError(f"No se encontró 'frogger_rl.py' a partir de {here}")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import frogger_rl  # helper del proyecto: se usa, no se replica

# Torch en CPU (igual que el proyecto original; no se asume CUDA) con un número
# modesto de hilos para que la máquina siga respondiendo durante el smoke.
import torch
import ale_py
import gymnasium
import stable_baselines3
import imageio
import matplotlib

DEVICE = "cpu"
torch.set_num_threads(max(1, min(4, os.cpu_count() or 4)))

RUNTIME = {
    "python": sys.version.split()[0],
    "python_executable": sys.executable,
    "numpy": np.__version__,
    "torch": torch.__version__,
    "gymnasium": gymnasium.__version__,
    "ale_py": ale_py.__version__,
    "stable_baselines3": stable_baselines3.__version__,
    "imageio": imageio.__version__,
    "matplotlib": matplotlib.__version__,
}
print("Modo smoke:", SMOKE)
print("Raíz del proyecto:", PROJECT_ROOT)
print(RUNTIME)

In [ ]:
# --- Configuración del experimento (modo normal y modo smoke) -----------------------
REQUESTED_STEPS = 100_000                     # presupuesto pedido (modo normal)
EVAL_EPISODES = 20                            # episodios held-out por política
EVAL_CAP = frogger_rl.ENV_MAX_AGENT_STEPS     # 2700 pasos de agente; NO 450
RANDOM_SEED = 424_242                         # RNG local del baseline aleatorio
EVAL_SEED_BASE = 777_001                      # semillas held-out (!= TRAIN_SEED)
EVAL_SEEDS = [EVAL_SEED_BASE + i for i in range(20)]

PPO_KWARGS = dict(
    n_steps=512,
    batch_size=64,
    n_epochs=4,
    learning_rate=2.5e-4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
)

if SMOKE:
    # FROGGER_SMOKE=1: un PPO real pero diminuto (plomería, no aprendizaje).
    # 256 pasos pedidos = 4 rollouts de 64 = 256 pasos reales exactos.
    REQUESTED_STEPS = 256
    EVAL_EPISODES = 2
    EVAL_CAP = 64
    PPO_KWARGS.update(n_steps=64, batch_size=64, n_epochs=1)

EVAL_SEEDS = EVAL_SEEDS[:EVAL_EPISODES]

# Guardas de semillas: errores claros ANTES de crear cualquier emulador.
if not isinstance(TRAIN_SEED, int) or isinstance(TRAIN_SEED, bool) or TRAIN_SEED < 0:
    raise ValueError(f"TRAIN_SEED debe ser un entero no negativo, no {TRAIN_SEED!r}")
if len(set(EVAL_SEEDS)) != len(EVAL_SEEDS):
    raise ValueError("las semillas held-out deben ser distintas entre si")
if min(EVAL_SEEDS) < 0:
    raise ValueError("las semillas held-out deben ser no negativas")
if TRAIN_SEED in EVAL_SEEDS:
    raise ValueError(
        "TRAIN_SEED coincide con una semilla held-out de evaluacion: "
        f"{TRAIN_SEED} en {EVAL_SEEDS}. Elija un TRAIN_SEED fuera de EVAL_SEEDS."
    )

print({
    "smoke": SMOKE,
    "train_seed": TRAIN_SEED,
    "requested_steps": REQUESTED_STEPS,
    "eval_episodes": len(EVAL_SEEDS),
    "eval_cap": EVAL_CAP,
    "eval_seeds": EVAL_SEEDS,
    "random_baseline_seed": RANDOM_SEED,
    "ppo_kwargs": PPO_KWARGS,
})

## Sondeo del entorno

Antes de entrenar, verificamos que el emulador responda con el contrato esperado:
observaciones `uint8 (4, 84, 84)`, **5 acciones**, un paso de `NOOP` y la recompensa
cruda. Todo dentro de `try/finally` para cerrar siempre el entorno.

In [ ]:
# Sondeo real de ALE/Frogger-v5 (entorno breve; cerrado en finally).
probe_env = frogger_rl.make_env(max_episode_steps=32)
try:
    obs, info = probe_env.reset(seed=(TRAIN_SEED % 10_000) + 1)
    print("observación:", obs.shape, obs.dtype,
          "min/max:", int(obs.min()), int(obs.max()))
    print("espacio de acciones:", probe_env.action_space)
    meanings = probe_env.unwrapped.get_action_meanings()
    print("significados:", list(meanings)[: len(frogger_rl.ACTION_NAMES)])
    obs2, reward, terminated, truncated, info2 = probe_env.step(0)  # NOOP
    print("paso NOOP -> recompensa cruda:", float(reward),
          "| terminated:", bool(terminated), "| truncated:", bool(truncated))
    print("formas de observación estables:", obs2.shape == obs.shape)
finally:
    probe_env.close()
print("Sondeo OK: 5 acciones y observaciones uint8 (4, 84, 84).")

## Baseline 1: política aleatoria

Una línea de referencia: sin aprender nada, ¿cuántos puntos crudos acumula una
política que elige acciones al azar (uniforme sobre las 5 acciones)? La descomposición `episodes / metrics` del
helper `evaluate_policy` nos da el retorno y largo **crudos por episodio** (con
`terminated` / `truncated` y semillas), junto con los resúmenes. La política aleatoria
usa un RNG **local y sembrado** (`random_seed`), por lo que es reproducible y no
depende del generador global.

In [ ]:
# Baseline aleatorio: determinista gracias a su RNG local (semilla separada).
random_result = frogger_rl.evaluate_policy(
    None,
    episode_seeds=EVAL_SEEDS,
    max_episode_steps=EVAL_CAP,
    random_seed=RANDOM_SEED,
)
m = random_result["metrics"]
print("episodios evaluados:", random_result["num_episodes"])
print("retorno bruto medio:", m["return"]["mean"], "+-", m["return"]["std"])
print("truncados:", m["truncated_count"], "| terminados:", m["terminated_count"])

## Configuración del entrenamiento (Monitor y callback)

- `make_vec_env` envuelve cada entorno en **una única capa de Monitor**
  (`filename=None`: solo memoria), que registra el retorno y el largo **crudos** de
  cada episodio completado. Todo fuera del helper.
- Un **callback** guarda, por cada episodio completado, el **tiempo real** de
  entrenamiento (`num_timesteps`) y el **retorno real**: de ahí sale la curva de
  aprendizaje; nunca inventamos muestras.
- `PPO(..., seed=TRAIN_SEED, ...)` **siembra el VecEnv de entrenamiento** al
  construirse y el aprendizaje arranca desde un **reset** explícito.
- Si la construcción, el reset o la evaluación previa fallan, el VecEnv de
  entrenamiento se cierra antes de propagar el error (guardas `try/except`).

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.env_util import make_vec_env

def build_train_env():
    # make_vec_env añade la única capa de Monitor (filename=None -> memoria),
    # que registra retorno/largo crudos por episodio; no se duplica el wrapper.
    return frogger_rl.make_env(max_episode_steps=EVAL_CAP)

class ProgressCallback(BaseCallback):
    """Guarda (timestep real, retorno real, largo real) por episodio completado."""

    def __init__(self):
        super().__init__(verbose=0)
        self.timesteps = []
        self.returns = []
        self.lengths = []

    def _on_step(self):
        for info in self.locals["infos"]:
            if "episode" in info:
                self.timesteps.append(int(self.num_timesteps))
                self.returns.append(float(info["episode"]["r"]))
                self.lengths.append(int(info["episode"]["l"]))
        return True

# Guarda de cierre: si construir el VecEnv, construir el PPO o resetear falla,
# cerramos el VecEnv y propagamos el error (nunca queda un emulador abierto).
train_vecenv = None
try:
    train_vecenv = make_vec_env(build_train_env, n_envs=1)

    # PPO con seed=TRAIN_SEED siembra el VecEnv de entrenamiento al construirse.
    model = PPO(
        "CnnPolicy",
        train_vecenv,
        seed=TRAIN_SEED,
        device=DEVICE,
        verbose=1,
        **PPO_KWARGS,
    )
    train_vecenv.reset()  # reset explícito ANTES de aprender
except BaseException:
    if train_vecenv is not None:
        train_vecenv.close()
    raise

progress = ProgressCallback()
print("PPO construido (todavía, sin entrenar). n_steps:", model.n_steps,
      "| batch_size:", model.batch_size, "| n_epochs:", model.n_epochs)

## Baseline 2: el PPO ANTES de aprender

Evaluamos la política **tal como quedó al construir el modelo** (pesos de
inicialización, sin `learn()`). Esto separa el efecto de *entrenar* del efecto de
*usar un PPO*: cualquier diferencia con la política aleatoria solo puede deberse a la
inicialización y a la arquitectura.

In [ ]:
# Baseline 2: el PPO sin entrenar, sobre las MISMAS semillas held-out.
# Se guarda una instantánea de los parámetros previos al aprendizaje y, si esta
# celda falla, se cierra el VecEnv de entrenamiento antes de propagar el error.
try:
    PARAMS_BEFORE = {
        key: value.detach().clone()
        for key, value in model.policy.state_dict().items()
    }
    before_result = frogger_rl.evaluate_policy(
        model.policy,
        episode_seeds=EVAL_SEEDS,
        max_episode_steps=EVAL_CAP,
    )
except BaseException:
    train_vecenv.close()
    raise

m = before_result["metrics"]
print("retorno bruto medio (PPO sin entrenar):", m["return"]["mean"], "+-", m["return"]["std"])
print("truncados:", m["truncated_count"], "| terminados:", m["terminated_count"])

## Entrenamiento PPO

Ejecutamos `learn()` con el presupuesto configurado. Notable: **SB3 colecta rollouts
completos**, así que los *pasos reales* pueden ser mayores (o redondeados) respecto de
los *pasos pedidos*; guardamos ambos por separado en los artefactos.

In [ ]:
# Entrenamiento (diminuto en smoke; 100000 pasos en modo normal).
# Se cuentan los pasos del optimizador observados y se compara la política final
# contra la instantánea previa: evidencia de que PPO actualizó pesos.
class _UpdateCounter:
    """Cuenta cada paso real del optimizador de PPO (sin interceptar nada)."""

    def __init__(self, optimizer):
        self._step = optimizer.step
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        return self._step(*args, **kwargs)

# On-policy algorithms exponen el optimizador en self.policy.optimizer
update_counter = _UpdateCounter(model.policy.optimizer)
model.policy.optimizer.step = update_counter

try:
    model.learn(
        total_timesteps=REQUESTED_STEPS,
        callback=progress,
        progress_bar=False,
    )
finally:
    train_vecenv.close()  # cerrar el VecEnv de entrenamiento en todas las rutas

ACTUAL_STEPS = int(model.num_timesteps)  # pasos reales colectados por SB3
OPTIMIZER_UPDATES = int(update_counter.count)

# Cambio máximo absoluto de parámetros respecto de la instantánea previa.
MAX_PARAM_DELTA = 0.0
for key, before_tensor in PARAMS_BEFORE.items():
    now_tensor = model.policy.state_dict()[key].detach()
    delta = float((now_tensor - before_tensor).abs().max().item())
    MAX_PARAM_DELTA = max(MAX_PARAM_DELTA, delta)
del PARAMS_BEFORE  # liberar la instantánea

print("pasos solicitados:", REQUESTED_STEPS)
print("pasos reales colectados:", ACTUAL_STEPS)
print("episodios completados (callback):", len(progress.returns))
print("pasos del optimizador observados:", OPTIMIZER_UPDATES)
print("cambio máximo de parámetros:", MAX_PARAM_DELTA)
print("primeros retornos crudos:", [round(r, 3) for r in progress.returns[:5]])

## Evaluación held-out DESPUÉS de aprender

Mismos episodios de evaluación (mismas semillas), misma tapa de pasos, mismo helper:
así las tres columnas (aleatorio, antes, después) son comparables. Las evaluaciones
abren y cierran **sus propios entornos**; en ninguna se confunde una truncación por
límite con el fin real del juego.

In [ ]:
# Evaluación final con el PPO entrenado, mismas semillas held-out.
after_result = frogger_rl.evaluate_policy(
    model.policy,
    episode_seeds=EVAL_SEEDS,
    max_episode_steps=EVAL_CAP,
)
m = after_result["metrics"]
print("retorno bruto medio (después):", m["return"]["mean"], "+-", m["return"]["std"])
print("truncados:", m["truncated_count"], "| terminados:", m["terminated_count"])
print("--- resumen de las tres evaluaciones ---")
for name, result, color in [
    ("aleatoria", random_result, None),
    ("antes", before_result, None),
    ("despues", after_result, None),
]:
    mm = result["metrics"]
    print(f"{name:9s} -> retorno medio {mm['return']['mean']:8.3f} "
          f"(n={result['num_episodes']}, trunc={mm['truncated_count']}, "
          f"term={mm['terminated_count']})")

## Artefactos de comparación (JSON y CSV)

Exportamos todo el contexto del experimento a `comparison.json` (serializable) y las
**filas por episodio + resúmenes** a `comparison.csv`: las tres evaluaciones genuinas,
ajustes del entorno/wrappers, semillas, conteos de truncación/terminación, bandera de
smoke, algoritmo/política/hiperparámetros y versiones de paquetes reales.

In [ ]:
import csv
import json

if OUTPUT_ROOT is None:
    OUTPUT_ROOT = PROJECT_ROOT / "results" / "frogger_ppo"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

COMPARISON = {
    "experiment": {
        "smoke": bool(SMOKE),
        "requested_timesteps": int(REQUESTED_STEPS),
        "actual_timesteps": int(ACTUAL_STEPS),
        "optimizer_updates": int(OPTIMIZER_UPDATES),
        "max_parameter_delta": float(MAX_PARAM_DELTA),
        "algorithm": "PPO",
        "policy": "CnnPolicy",
        "hyperparameters": dict(PPO_KWARGS),
        "device": DEVICE,
        "train_seed": int(TRAIN_SEED),
        "random_baseline_seed": int(RANDOM_SEED),
        "eval_episodes": int(len(EVAL_SEEDS)),
        "eval_seeds": [int(s) for s in EVAL_SEEDS],
        "eval_cap_agent_steps": int(EVAL_CAP),
    },
    "environment": {
        "env_id": frogger_rl.ENV_ID,
        "action_names": list(frogger_rl.ACTION_NAMES),
        "obs_shape": [int(v) for v in frogger_rl.OBS_SHAPE],
        "base_frameskip": int(frogger_rl.BASE_FRAMESKIP),
        "sticky_action_probability": float(frogger_rl.STICKY_ACTION_PROBABILITY),
        "preprocess_frame_skip": int(frogger_rl.PREPROCESS_FRAME_SKIP),
        "screen_size": 84,
        "grayscale_obs": True,
        "scale_obs": False,
        "terminal_on_life_loss": False,
        "stack_size": int(frogger_rl.STACK_SIZE),
        "reward_clipping": False,
        "reward_shaping": False,
    },
    "runtime_versions": dict(RUNTIME),
    "evaluations": {
        "random": random_result,
        "ppo_before_learn": before_result,
        "ppo_after_learn": after_result,
    },
}
with open(OUTPUT_ROOT / "comparison.json", "w", encoding="utf-8") as fh:
    json.dump(COMPARISON, fh, indent=2)

# CSV: una fila por episodio (retorno/largo crudos + flags + semilla) y resúmenes.
EPISODE_FIELDS = ["record_type", "policy", "episode", "seed", "return", "length",
                  "terminated", "truncated"]
SUMMARY_FIELDS = ["record_type", "policy", "n", "return_mean", "return_std",
                  "return_min", "return_max", "length_mean", "length_std",
                  "terminated_count", "truncated_count"]

rows = []
for key, label in [("random", "random"),
                   ("ppo_before_learn", "ppo_before_learn"),
                   ("ppo_after_learn", "ppo_after_learn")]:
    result = COMPARISON["evaluations"][key]
    for ep in result["episodes"]:
        rows.append({
            "record_type": "episode", "policy": label,
            "episode": int(ep["episode"]), "seed": int(ep["seed"]),
            "return": float(ep["return"]), "length": int(ep["length"]),
            "terminated": int(bool(ep["terminated"])),
            "truncated": int(bool(ep["truncated"])),
        })
    mm = result["metrics"]
    rows.append({
        "record_type": "summary", "policy": label, "n": int(mm["n"]),
        "return_mean": float(mm["return"]["mean"]),
        "return_std": float(mm["return"]["std"]),
        "return_min": float(mm["return"]["min"]),
        "return_max": float(mm["return"]["max"]),
        "length_mean": float(mm["length"]["mean"]),
        "length_std": float(mm["length"]["std"]),
        "terminated_count": int(mm["terminated_count"]),
        "truncated_count": int(mm["truncated_count"]),
    })

header = list(dict.fromkeys(EPISODE_FIELDS + SUMMARY_FIELDS))
with open(OUTPUT_ROOT / "comparison.csv", "w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=header)
    writer.writeheader()
    writer.writerows(rows)

print("comparison.json y comparison.csv escritos en", OUTPUT_ROOT)

## Curva de aprendizaje

Graficamos el **retorno crudo** de cada episodio completado contra los **pasos reales**
de entrenamiento (datos genuinos del callback), con una línea de media móvil y los
niveles de referencia: baseline aleatorio y PPO sin entrenar. Si un presupuesto
diminuto no completara ningún episodio, el gráfico lo dice explícitamente en vez de
inventar muestras.

In [ ]:
import matplotlib.pyplot as plt

def moving_average(values, window=5):
    if not values:
        return []
    window = max(1, min(window, len(values)))
    return [float(np.mean(values[max(0, i - window + 1): i + 1]))
            for i in range(len(values))]

curve_rows = [
    {"timestep": int(t), "episode_return": float(r), "episode_length": int(l)}
    for t, r, l in zip(progress.timesteps, progress.returns, progress.lengths)
]
with open(OUTPUT_ROOT / "curve.csv", "w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=["timestep", "episode_return", "episode_length"])
    writer.writeheader()
    writer.writerows(curve_rows)

random_mean = float(random_result["metrics"]["return"]["mean"])
before_mean = float(before_result["metrics"]["return"]["mean"])

fig, ax = plt.subplots(figsize=(9, 5))
if curve_rows:
    ts = [r["timestep"] for r in curve_rows]
    rs = [r["episode_return"] for r in curve_rows]
    ax.plot(ts, rs, ".", color="#1f77b4", label="retorno crudo por episodio")
    ax.plot(ts, moving_average(rs), "-", color="#d62728", alpha=0.9, label="media móvil")
    ax.axhline(random_mean, linestyle="--", color="#2ca02c",
               label=f"baseline aleatorio = {random_mean:.3g}")
    ax.axhline(before_mean, linestyle=":", color="#9467bd",
               label=f"PPO sin entrenar = {before_mean:.3g}")
    ax.set_xlabel("pasos reales de entrenamiento")
    ax.set_ylabel("retorno bruto por episodio")
    ax.set_title("Curva de aprendizaje (retorno crudo; sin datos inventados)")
    ax.legend()
else:
    ax.text(0.5, 0.5, "no completed episode yet",
            ha="center", va="center", transform=ax.transAxes, fontsize=14)
    ax.set_title("Curva de aprendizaje: sin episodios completados en este presupuesto")
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "curve.png", dpi=120)
plt.close(fig)
print("curve.csv:", len(curve_rows), "filas | curve.png guardado en", OUTPUT_ROOT)

## Incertidumbre e interpretación honesta

- **Modo smoke (`FROGGER_SMOKE=1`)** ejecuta un PPO real pero diminuto: 256 pasos,
  4 rollouts, 2 episodios de evaluación por política y un GIF breve. Eso prueba la
  **plomería** (flujo de observaciones, actualización de gradientes, exportación de
  artefactos), **no** que la política aprenda.
- En **modo normal** el retorno por episodio es muy variable: 20 episodios por
  política sirven como referencia, pero para conclusiones firmes conviene repetir el
  entrenamiento con **varias semillas** (`TRAIN_SEED` variable de entorno) y comparar
  las distribuciones de retornos, no solo las medias.
- `truncated` **no** es éxito: un episodio cortado por el límite de pasos solo
  significa que el emulador seguía vivo al llegar al tope.
- Un retorno crudo igual a 0 en presupuestos cortos es un dato real, no un error.
- El notebook reporta los pasos del optimizador **observados** (`optimizer_updates`)
  y el mayor cambio absoluto de pesos (`max_parameter_delta`). Valores positivos
  prueban que PPO actualizó pesos; "hubo actualizaciones" **no** es "hay aprendizaje
  útil".

## Guardar y recargar el modelo

Exportamos `model.zip` (pesos + configuración), recargamos el zip con `PPO.load` y
comprobamos que sobre una **observación fija** la acción determinista coincide antes y
después de cargar. El zip nace en **esta sesión** (no se recargan pesos previos de
ningún otro proyecto, p. ej. Airstriker/Genesis).

In [ ]:
MODEL_PATH = OUTPUT_ROOT / "model.zip"
fixed_env = frogger_rl.make_env(max_episode_steps=16)
try:
    fixed_obs, _ = fixed_env.reset(seed=902_911)  # observación fija y reproducible

    def deterministic_action(policy_obj):
        act, _ = policy_obj.predict(fixed_obs, deterministic=True)
        return int(np.asarray(act).reshape(-1)[0])

    action_before_save = deterministic_action(model)
    model.save(str(MODEL_PATH))
    loaded_model = PPO.load(str(MODEL_PATH), device=DEVICE)
    action_after_load = deterministic_action(loaded_model)
    RELOAD_OK = action_before_save == action_after_load
    print("acción determinista antes de guardar:", action_before_save)
    print("acción determinista tras cargar:", action_after_load)
    print("coinciden:", RELOAD_OK, "| zip:", MODEL_PATH)
finally:
    fixed_env.close()

## GIF corto de un episodio

Renderizamos `rgb_array` y guardamos un GIF RGB breve (cantidad acotada de cuadros)
con la política recargada. En smoke el GIF queda corto a propósito: suficiente para
comprobar que la grabación funciona, sin prometer juego aprendido.

In [ ]:
MAX_GIF_FRAMES = 20 if SMOKE else 120
GIF_PATH = OUTPUT_ROOT / ("episodio_smoke.gif" if SMOKE else "episodio_ppo.gif")

gif_env = frogger_rl.make_env(max_episode_steps=EVAL_CAP, render_mode="rgb_array")
frames = []
try:
    obs, _ = gif_env.reset(seed=EVAL_SEED_BASE)
    frames.append(np.asarray(gif_env.render(), dtype=np.uint8))
    for _ in range(MAX_GIF_FRAMES):
        if len(frames) >= MAX_GIF_FRAMES:
            break
        action, _ = loaded_model.predict(obs, deterministic=True)
        act = int(np.asarray(action).reshape(-1)[0])
        obs, _reward, terminated, truncated, _ = gif_env.step(act)
        frames.append(np.asarray(gif_env.render(), dtype=np.uint8))
        if terminated or truncated:
            break
finally:
    gif_env.close()

imageio.mimsave(str(GIF_PATH), frames, format="GIF", duration=0.12)
print("GIF RGB guardado:", GIF_PATH, "| cuadros:", len(frames))

ARTIFACTS = {
    "model_zip": MODEL_PATH,
    "curve_csv": OUTPUT_ROOT / "curve.csv",
    "curve_png": OUTPUT_ROOT / "curve.png",
    "comparison_json": OUTPUT_ROOT / "comparison.json",
    "comparison_csv": OUTPUT_ROOT / "comparison.csv",
    "gif": GIF_PATH,
}
MANIFEST = {
    "output_root": str(OUTPUT_ROOT),
    "smoke": bool(SMOKE),
    "requested_timesteps": int(REQUESTED_STEPS),
    "actual_timesteps": int(ACTUAL_STEPS),
    "completed_episodes_training": int(len(progress.returns)),
    "reload_matches": bool(RELOAD_OK),
    "optimizer_updates": int(OPTIMIZER_UPDATES),
    "max_parameter_delta": float(MAX_PARAM_DELTA),
    "gif_frames": int(len(frames)),
    "runtime": dict(RUNTIME),
    "artifacts": {name: str(path) for name, path in ARTIFACTS.items()},
    "manifest_json": str(OUTPUT_ROOT / "manifest.json"),
}
with open(OUTPUT_ROOT / "manifest.json", "w", encoding="utf-8") as fh:
    json.dump(MANIFEST, fh, indent=2)

print("=== MANIFEST ===")
print(json.dumps(MANIFEST, indent=2))